In [1]:
from nichenetpy.prediction import LigandActivityPredictor, LigandReceptorNetwork
from nichenetpy.utils import read_matrix_from_csv
from nichenetpy.extraction import get_expressed_genes, subset_ann_celltype

import anndata
import scanpy as sc

In [2]:
predictor = LigandActivityPredictor(*read_matrix_from_csv("D:/Data/nichenetpy/testargs/ligand_target_matrix.csv"))
lr_network = LigandReceptorNetwork(filename="D:/Data/nichenetpy/csv/lr_network.csv")
ann = anndata.io.read_h5ad("D:/Data/nichenetpy/annData/annData3531889.h5")
ann

AnnData object with n_obs × n_vars = 5027 × 13541
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'nGene', 'nUMI', 'aggregate', 'res.0.6', 'celltype'
    var: 'gene'
    layers: 'counts', 'data', 'scale.data'

In [3]:
celltype_counts = ann.obs["celltype"].value_counts()
celltype_counts

celltype
CD4 T    2562
CD8 T    1645
B         382
Treg      199
NK        131
Mono       90
DC         18
Name: count, dtype: int64

In [4]:
receiver = "CD8 T"
expressed_genes_receiver = set(get_expressed_genes(receiver, ann, 0.05))
all_receptors = lr_network.get_receptors()
expressed_receptors = all_receptors.intersection(expressed_genes_receiver)
potential_ligands = set(key for key, group in lr_network.item_iter() if len(group.intersection(expressed_receptors)) > 0)

In [5]:
sender_celltypes = ("CD4 T", "Treg", "Mono", "NK", "B", "DC")
list_expressed_genes_sender = [get_expressed_genes(ct, ann, pct=0.05) for ct in sender_celltypes]
expressed_genes_sender = set(e for l in list_expressed_genes_sender for e in l)
potential_ligands_focused = potential_ligands.intersection(expressed_genes_sender)

In [6]:
ann_receiver = subset_ann_celltype(ann, receiver, layers=["data"])
sc.pp.log1p(ann_receiver, layer="data")
sc.tl.rank_genes_groups(ann_receiver, groupby="aggregate", method="wilcoxon", layer="data")
geneset = [
    ann.var["gene"].iloc[int(gene)] for gene, pval_adj, log2FC in
    zip(
        [e[0] for e in ann_receiver.uns["rank_genes_groups"]["names"]],
        [e[0] for e in ann_receiver.uns["rank_genes_groups"]["pvals_adj"]],
        [e[0] for e in ann_receiver.uns["rank_genes_groups"]["logfoldchanges"]]
    ) if pval_adj <= 0.05 and abs(log2FC) >= 0.25
]

In [17]:
ligand_activities = predictor.predict_ligand_activities(
    geneset=geneset,
    background_expressed_genes=expressed_genes_receiver,
    potential_ligands=potential_ligands
)
ligand_activities = sorted(ligand_activities.items(), key=lambda x : x[1]["aupr_corrected"], reverse=True)
ligand_activities[:10]

[('Ifna1',
  {'aupr': np.float64(0.4084763950778146),
   'aupr_corrected': np.float64(0.3462513793644268)}),
 ('Ifnl3',
  {'aupr': np.float64(0.3804891531351264),
   'aupr_corrected': np.float64(0.3182641374217386)}),
 ('Ifnb1',
  {'aupr': np.float64(0.35312166696938047),
   'aupr_corrected': np.float64(0.2908966512559927)}),
 ('Il27',
  {'aupr': np.float64(0.34806690186419736),
   'aupr_corrected': np.float64(0.2858418861508096)}),
 ('Ifng',
  {'aupr': np.float64(0.32299961267633875),
   'aupr_corrected': np.float64(0.26077459696295097)}),
 ('Ifnk',
  {'aupr': np.float64(0.2461579543171219),
   'aupr_corrected': np.float64(0.18393293860373408)}),
 ('Ifne',
  {'aupr': np.float64(0.24416429204346932),
   'aupr_corrected': np.float64(0.1819392763300815)}),
 ('Ebi3',
  {'aupr': np.float64(0.23133604768919744),
   'aupr_corrected': np.float64(0.16911103197580962)}),
 ('Ifnl2',
  {'aupr': np.float64(0.22058860108693226),
   'aupr_corrected': np.float64(0.15836358537354445)}),
 ('Ifna2',
  {